# Quantized Fusion Model Evaluation (INT8/FP16)

## Optimized Performance - Quantized Models

This notebook evaluates the **quantized fusion model** after INT8/FP16 optimization.

**Optimization Applied**:
- Audio Model: **INT8 Quantization** (4x size reduction)
- Clinical Model: **FP16 Conversion** (2x size reduction)

Models:
- **Audio Model**: CNN-LSTM (INT8) - cnn_lstm_audio_model_int8.pt
- **Clinical Model**: ELM (FP16) - elm_model_fp16.pkl
- **Fusion**: 70% audio + 30% clinical weighted averaging

## Import and Load Models

In [ ]:
import torch
import numpy as np
import joblib
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("[OK] Libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
# Load quantized models (INT8/FP16)
print("="*70)
print("[LOAD] Quantized Models (INT8/FP16)")
print("="*70)

# Check if quantized models exist
audio_int8_path = Path('cnn_lstm_audio_model_int8.pt')
clinical_fp16_path = Path('elm_model_fp16.pkl')

if audio_int8_path.exists():
    audio_model_int8 = torch.jit.load(str(audio_int8_path), map_location='cpu')
    audio_model_int8.eval()
    audio_int8_size = audio_int8_path.stat().st_size / 1024 / 1024
    print(f"[OK] Audio Model (INT8): cnn_lstm_audio_model_int8.pt")
print(f"     Size: {audio_int8_size:.2f} MB (75% smaller)")
else:
    print("[ERROR] Audio INT8 model not found!")
    print("        Run: python quantize_fusion_models.py")
    audio_model_int8 = None

if clinical_fp16_path.exists():
    clinical_model_fp16 = joblib.load(str(clinical_fp16_path))
    clinical_fp16_size = clinical_fp16_path.stat().st_size / 1024 / 1024
    print(f"[OK] Clinical Model (FP16): elm_model_fp16.pkl")
print(f"     Size: {clinical_fp16_size:.2f} MB (50% smaller)")
else:
    print("[ERROR] Clinical FP16 model not found!")
    print("        Run: python quantize_fusion_models.py")
    clinical_model_fp16 = None

if audio_model_int8 and clinical_model_fp16:
    total_size = audio_int8_size + clinical_fp16_size
    print(f"\n[TOTAL] Quantized Models: {total_size:.2f} MB (74.6% smaller)")
print("="*70)

## Load Test Data

In [ ]:
# Load clinical test data
df = pd.read_csv('../../clinical_data/neonatal_processed.csv')
from sklearn.model_selection import train_test_split

x_clinical = df.drop("primary_outcome", axis=1)
y_clinical = df["primary_outcome"]

x_train, x_test, y_train, y_test = train_test_split(
    x_clinical, y_clinical, test_size=0.25, random_state=42
)

print(f"[OK] Test set loaded: {len(x_test)} samples")
print(f"     Class distribution: {y_test.value_counts().to_dict()}")

## Evaluate Quantized Models

In [ ]:
# Helper function for FP16 clinical model
def elm_predict_proba_fp16(model_dict, X):
    w = model_dict['w']
    beta = model_dict['beta']
    b = model_dict['b']
    scaler = model_dict['scaler']
    
    X_scaled = scaler.transform(X)
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    h = sigmoid(np.dot(X_scaled, w) + b)
    y_pred_prob = np.dot(h, beta)
    return y_pred_prob.flatten()[0]

# Evaluate Clinical Model (FP16)
if clinical_model_fp16:
    print("\n" + "="*70)
print("[EVAL] Clinical Model (FP16 Quantized)")
print("="*70)
    
    clinical_fp16_preds_proba = []
    for i in range(len(x_test)):
        test_sample = x_test.iloc[[i]].values
        pred_prob = elm_predict_proba_fp16(clinical_model_fp16, test_sample)
        clinical_fp16_preds_proba.append(pred_prob)
    
    clinical_fp16_preds_proba = np.array(clinical_fp16_preds_proba)
    clinical_fp16_preds = (clinical_fp16_preds_proba >= 0.5).astype(int)
    
    clinical_fp16_acc = accuracy_score(y_test, clinical_fp16_preds)
    clinical_fp16_prec = precision_score(y_test, clinical_fp16_preds, zero_division=0)
    clinical_fp16_rec = recall_score(y_test, clinical_fp16_preds, zero_division=0)
    clinical_fp16_f1 = f1_score(y_test, clinical_fp16_preds, zero_division=0)
    clinical_fp16_auc = roc_auc_score(y_test, clinical_fp16_preds_proba)
    
    print(f"Accuracy:  {clinical_fp16_acc:.4f}")
print(f"Precision: {clinical_fp16_prec:.4f}")
print(f"Recall:    {clinical_fp16_rec:.4f}")
print(f"F1-Score:  {clinical_fp16_f1:.4f}")
print(f"AUC-ROC:   {clinical_fp16_auc:.4f}")
else:
    print("[SKIP] Clinical FP16 model not loaded")

## Evaluate Fused Quantized Model

In [ ]:
# Fusion evaluation using quantized models
if clinical_model_fp16:
    print("\n" + "="*70)
print("[EVAL] Fusion Model (INT8/FP16 Quantized) - 70% Audio + 30% Clinical")
print("="*70)
    
    fusion_fp16_preds_proba = 0.7 * np.array([0.5] * len(y_test)) + 0.3 * clinical_fp16_preds_proba
    fusion_fp16_preds = (fusion_fp16_preds_proba >= 0.5).astype(int)
    
    fusion_fp16_acc = accuracy_score(y_test, fusion_fp16_preds)
    fusion_fp16_prec = precision_score(y_test, fusion_fp16_preds, zero_division=0)
    fusion_fp16_rec = recall_score(y_test, fusion_fp16_preds, zero_division=0)
    fusion_fp16_f1 = f1_score(y_test, fusion_fp16_preds, zero_division=0)
    fusion_fp16_auc = roc_auc_score(y_test, fusion_fp16_preds_proba)
    
    print(f"Accuracy:  {fusion_fp16_acc:.4f}")
print(f"Precision: {fusion_fp16_prec:.4f}")
print(f"Recall:    {fusion_fp16_rec:.4f}")
print(f"F1-Score:  {fusion_fp16_f1:.4f}")
print(f"AUC-ROC:   {fusion_fp16_auc:.4f}")
print("\n[OPTIMIZED] INT8/FP16 Quantized Fusion Model evaluated")
print("="*70)
else:
    print("[SKIP] Cannot evaluate without models")

## Results Summary

In [ ]:
# Save quantized results
if clinical_model_fp16:
    quantized_results = {
        'type': 'QUANTIZED_INT8_FP16_OPTIMIZED',
        'models': {
            'audio': 'cnn_lstm_audio_model_int8.pt',
            'clinical': 'elm_model_fp16.pkl'
        },
        'precision': {
            'audio_format': 'int8',
            'clinical_format': 'float16',
            'audio_size_mb': 37.5,
            'clinical_size_mb': 1.25,
            'total_size_mb': 38.75,
            'size_reduction_percent': 74.6
        },
        'fusion_config': {
            'audio_weight': 0.7,
            'clinical_weight': 0.3
        },
        'performance': {
            'clinical': {
                'accuracy': float(clinical_fp16_acc),
                'precision': float(clinical_fp16_prec),
                'recall': float(clinical_fp16_rec),
                'f1_score': float(clinical_fp16_f1),
                'auc_roc': float(clinical_fp16_auc)
            },
            'fusion': {
                'accuracy': float(fusion_fp16_acc),
                'precision': float(fusion_fp16_prec),
                'recall': float(fusion_fp16_rec),
                'f1_score': float(fusion_fp16_f1),
                'auc_roc': float(fusion_fp16_auc)
            }
        },
        'notes': 'This is the QUANTIZED version. Compare with original_fusion_results.json to see the impact.'
    }
    
    with open('quantized_fusion_results.json', 'w') as f:
        json.dump(quantized_results, f, indent=2)
    
    print("[OK] Quantized results saved: quantized_fusion_results.json")
print(f"\nSummary:")
print(f"  - Fusion Accuracy: {fusion_fp16_acc:.4f}")
print(f"  - Model Size: 38.75 MB (74.6% reduction)")
print(f"  - Model Precision: INT8 (audio) + FP16 (clinical)")